In [4]:
!pip install gradio


[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: C:\Users\pra72\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
"""
Interactive Plant Disease Classification Interface
-------------------------------------------------
This code extends the ResNet9 plant disease classifier with an interactive interface
for uploading leaf images and predicting diseases with probability percentages.
"""

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import io
# import base64 # Not used, can be removed
import gradio as gr

# --- MODEL ARCHITECTURE (Identical to yours, kept for completeness) ---
class ImageClassificationBase(nn.Module):
    def training_step(self, batch):
        images, labels = batch
        out = self(images)
        loss = F.cross_entropy(out, labels)
        return loss

    def validation_step(self, batch):
        images, labels = batch
        out = self(images)
        loss = F.cross_entropy(out, labels)
        acc = accuracy(out, labels)
        return {"val_loss": loss.detach(), "val_accuracy": acc}

    def validation_epoch_end(self, outputs):
        batch_losses = [x["val_loss"] for x in outputs]
        batch_accuracy = [x["val_accuracy"] for x in outputs]
        epoch_loss = torch.stack(batch_losses).mean()
        epoch_accuracy = torch.stack(batch_accuracy).mean()
        return {"val_loss": epoch_loss, "val_accuracy": epoch_accuracy}

    def epoch_end(self, epoch, result):
        # This function might not be called in inference-only mode,
        # but keeping it for consistency with the model definition.
        if 'lrs' in result and result['lrs']: # Check if lrs is available and not empty
            print(f"Epoch [{epoch}], last_lr: {result['lrs'][-1]:.5f}, train_loss: {result.get('train_loss', 'N/A'):.4f}, val_loss: {result['val_loss']:.4f}, val_acc: {result['val_accuracy']:.4f}")
        else:
            print(f"Epoch [{epoch}], train_loss: {result.get('train_loss', 'N/A'):.4f}, val_loss: {result['val_loss']:.4f}, val_acc: {result['val_accuracy']:.4f}")


def ConvBlock(in_channels, out_channels, pool=False):
    layers = [
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True)
    ]
    if pool:
        layers.append(nn.MaxPool2d(4))
    return nn.Sequential(*layers)

class ResNet9(ImageClassificationBase):
    def __init__(self, in_channels, num_diseases):
        super().__init__()
        self.conv1 = ConvBlock(in_channels, 64)
        self.conv2 = ConvBlock(64, 128, pool=True)
        self.res1 = nn.Sequential(ConvBlock(128, 128), ConvBlock(128, 128))
        self.conv3 = ConvBlock(128, 256, pool=True)
        self.conv4 = ConvBlock(256, 512, pool=True)
        self.res2 = nn.Sequential(ConvBlock(512, 512), ConvBlock(512, 512))
        self.classifier = nn.Sequential(
            nn.MaxPool2d(4),
            nn.Flatten(),
            nn.Linear(512, num_diseases)
        )
    def forward(self, xb):
        out = self.conv1(xb)
        out = self.conv2(out)
        out = self.res1(out) + out
        out = self.conv3(out)
        out = self.conv4(out)
        out = self.res2(out) + out
        out = self.classifier(out)
        return out

# --- HELPER FUNCTIONS ---
def accuracy(outputs, labels):
    _, preds = torch.max(outputs, dim=1)
    return torch.tensor(torch.sum(preds == labels).item() / len(preds))

def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

def to_device(data, device):
    if isinstance(data, (list, tuple)):
        return [to_device(x, device) for x in data]
    return data.to(device, non_blocking=True)

# --- PREDICTION FUNCTIONS ---
def load_model_state(model_path, num_classes):
    device = get_device()
    model = ResNet9(3, num_classes)
    try:
        model.load_state_dict(torch.load(model_path, map_location=device))
        print(f"Model state loaded successfully from {model_path}")
    except FileNotFoundError:
        print(f"Error: Model file not found at {model_path}. Using an uninitialized model (predictions will be random).")
    except Exception as e:
        print(f"Error loading model state: {e}. Using an uninitialized model (predictions will be random).")
    model = to_device(model, device)
    model.eval()
    return model

def process_image_for_prediction(image_input):
    if isinstance(image_input, str):
        pil_image = Image.open(image_input)
    elif isinstance(image_input, bytes):
        pil_image = Image.open(io.BytesIO(image_input))
    elif isinstance(image_input, Image.Image):
        pil_image = image_input
    else:
        raise ValueError(f"Unsupported image input type: {type(image_input)}")

    if pil_image.mode != 'RGB':
        pil_image = pil_image.convert('RGB')

    transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    return transform(pil_image), pil_image

def predict_top_k_diseases(img_tensor, model, class_names_list, device, top_k=3):
    xb = to_device(img_tensor.unsqueeze(0), device)
    model.eval()
    with torch.no_grad():
        outputs = model(xb)
        probabilities = F.softmax(outputs, dim=1)[0]
        topk_probs, topk_indices = torch.topk(probabilities, top_k)
        topk_probs_percent = topk_probs.cpu().numpy() * 100
        topk_indices_list = topk_indices.cpu().numpy()
        topk_classes_names = [class_names_list[idx] for idx in topk_indices_list]
        return list(zip(topk_classes_names, topk_probs_percent))

def format_display_disease_name(disease_class_str):
    parts = disease_class_str.split('___')
    plant = parts[0].replace('_', ' ')
    if len(parts) > 1:
        disease = parts[1].replace('_', ' ')
        return f"{plant} - {disease}" if disease != 'healthy' else f"{plant} (Healthy)"
    return plant

# --- INTERFACE FUNCTIONS ---
def create_prediction_visualization(original_pil_image, predictions_list):
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(original_pil_image.resize((256,256)))
    plt.title("Uploaded Image")
    plt.axis('off')

    plt.subplot(1, 2, 2)
    names = [pred[0] for pred in predictions_list]
    probs = [pred[1] for pred in predictions_list]
    colors = ['#4CAF50', '#2196F3', '#FFC107', '#FF9800', '#F44336'][:len(names)]

    y_pos = np.arange(len(names))
    bars = plt.barh(y_pos, probs, color=colors, align='center')
    plt.yticks(y_pos, names)
    plt.gca().invert_yaxis()
    plt.title(f"Top {len(names)} Predictions")
    plt.xlabel("Probability (%)")
    plt.xlim(0, 105)

    for i, bar in enumerate(bars):
        plt.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2.,
                 f"{probs[i]:.1f}%", va='center', ha='left')

    plt.tight_layout()
    buf = io.BytesIO()
    plt.savefig(buf, format='png', bbox_inches='tight')
    plt.close()
    buf.seek(0)
    return Image.open(buf)

def generate_prediction_outputs(image_input, model_instance, class_names_list, device):
    try:
        img_tensor, original_pil_image = process_image_for_prediction(image_input)
        raw_predictions = predict_top_k_diseases(img_tensor, model_instance, class_names_list, device, top_k=3)

        formatted_predictions = []
        result_text_lines = [f"Top {len(raw_predictions)} Disease Predictions:"]
        for i, (class_name, probability) in enumerate(raw_predictions):
            display_name = format_display_disease_name(class_name)
            formatted_predictions.append((display_name, probability))
            result_text_lines.append(f"{i+1}. {display_name}: {probability:.1f}%")

        visualization_pil_image = create_prediction_visualization(original_pil_image, formatted_predictions)
        return visualization_pil_image, "\n".join(result_text_lines)

    except Exception as e:
        print(f"Error during prediction: {e}")
        # Log the full traceback for debugging if needed
        # import traceback
        # print(traceback.format_exc())
        return None, f"An error occurred: {str(e)}. Please check the image format or try another image."

# --- MAIN CLASS ---
class PlantDiseasePredictor:
    def __init__(self, model_path=None, class_names_path=None):
        self.device = get_device()
        print(f"Using device: {self.device}")
        self.num_classes = 38
        self.class_names = [
            "Apple___Apple_scab", "Apple___Black_rot", "Apple___Cedar_apple_rust",
            "Apple___healthy", "Background_without_leaves", "Blueberry___healthy",
            "Cherry___Powdery_mildew", "Cherry___healthy", "Corn___Cercospora_leaf_spot Gray_leaf_spot",
            "Corn___Common_rust", "Corn___Northern_Leaf_Blight", "Corn___healthy",
            "Grape___Black_rot", "Grape___Esca_(Black_Measles)", "Grape___Leaf_blight_(Isariopsis_Leaf_Spot)",
            "Grape___healthy", "Orange___Haunglongbing_(Citrus_greening)", "Peach___Bacterial_spot",
            "Peach___healthy", "Pepper,_bell___Bacterial_spot", "Pepper,_bell___healthy",
            "Potato___Early_blight", "Potato___Late_blight", "Potato___healthy",
            "Raspberry___healthy", "Soybean___healthy", "Squash___Powdery_mildew",
            "Strawberry___Leaf_scorch", "Strawberry___healthy", "Tomato___Bacterial_spot",
            "Tomato___Early_blight", "Tomato___Late_blight", "Tomato___Leaf_Mold",
            "Tomato___Septoria_leaf_spot", "Tomato___Spider_mites Two-spotted_spider_mite",
            "Tomato___Target_Spot", "Tomato___Tomato_Yellow_Leaf_Curl_Virus", "Tomato___healthy"
        ]

        if class_names_path and os.path.exists(class_names_path):
            try:
                with open(class_names_path, 'r') as f:
                    loaded_classes = [line.strip() for line in f.readlines() if line.strip()]
                if len(loaded_classes) == self.num_classes:
                    self.class_names = loaded_classes
                    print(f"Loaded {len(self.class_names)} classes from {class_names_path}")
                else:
                    print(f"Warning: Class names file has {len(loaded_classes)} classes, model expects {self.num_classes}. Using default classes.")
            except Exception as e:
                print(f"Error loading class names from {class_names_path}: {e}. Using default classes.")
        else:
            if class_names_path: print(f"Class names file not found: {class_names_path}. Using default classes.")
            else: print(f"Using default list of {len(self.class_names)} classes.")

        self.model = load_model_state(model_path, self.num_classes)

    def predict(self, image_input_for_gradio):
        return generate_prediction_outputs(image_input_for_gradio, self.model, self.class_names, self.device)

# --- GRADIO INTERFACE ---
def create_gradio_interface(predictor_instance):
    def interface_fn_for_gradio(image_filepath_from_gradio):
        pil_image_output, text_output = predictor_instance.predict(image_filepath_from_gradio)
        return pil_image_output, text_output

    interface = gr.Interface(
        fn=interface_fn_for_gradio,
        inputs=gr.Image(type="filepath", label="Upload Leaf Image"),
        outputs=[
            gr.Image(type="pil", label="Prediction Results"),
            gr.Textbox(label="Detailed Results", lines=5)
        ],
        title="🌿 Plant Disease Diagnosis System 🌿",
        description="Upload an image of a plant leaf to diagnose potential diseases. "
                    "The system shows the top 3 most likely diseases with probability percentages.",
        examples=None, # REMOVED EXAMPLES
        theme=gr.themes.Soft(),
        allow_flagging="never"
    )
    return interface

# --- MAIN EXECUTION (GRADIO) ---
def run_gradio_app():
    print("=" * 50)
    print("PLANT DISEASE INTERACTIVE PREDICTOR (GRADIO)")
    print("=" * 50)

    model_file = "best_plant_disease_model.pth"
    class_names_file = "class_names.txt"

    if not os.path.exists(class_names_file):
        print(f"'{class_names_file}' not found. Creating with default class names.")
        default_classes = [
            "Apple___Apple_scab", "Apple___Black_rot", "Apple___Cedar_apple_rust",
            "Apple___healthy", "Background_without_leaves", "Blueberry___healthy",
            "Cherry___Powdery_mildew", "Cherry___healthy", "Corn___Cercospora_leaf_spot Gray_leaf_spot",
            "Corn___Common_rust", "Corn___Northern_Leaf_Blight", "Corn___healthy",
            "Grape___Black_rot", "Grape___Esca_(Black_Measles)", "Grape___Leaf_blight_(Isariopsis_Leaf_Spot)",
            "Grape___healthy", "Orange___Haunglongbing_(Citrus_greening)", "Peach___Bacterial_spot",
            "Peach___healthy", "Pepper,_bell___Bacterial_spot", "Pepper,_bell___healthy",
            "Potato___Early_blight", "Potato___Late_blight", "Potato___healthy",
            "Raspberry___healthy", "Soybean___healthy", "Squash___Powdery_mildew",
            "Strawberry___Leaf_scorch", "Strawberry___healthy", "Tomato___Bacterial_spot",
            "Tomato___Early_blight", "Tomato___Late_blight", "Tomato___Leaf_Mold",
            "Tomato___Septoria_leaf_spot", "Tomato___Spider_mites Two-spotted_spider_mite",
            "Tomato___Target_Spot", "Tomato___Tomato_Yellow_Leaf_Curl_Virus", "Tomato___healthy"
        ]
        try:
            with open(class_names_file, 'w') as f:
                for cls_name in default_classes:
                    f.write(cls_name + '\n')
            print(f"Created '{class_names_file}' with {len(default_classes)} classes.")
        except Exception as e:
            print(f"Could not create '{class_names_file}': {e}")

    predictor = PlantDiseasePredictor(model_path=model_file, class_names_path=class_names_file)
    gradio_ui = create_gradio_interface(predictor)
    print("\nLaunching Gradio interface... (Access it at the URL provided by Gradio, e.g., http://127.0.0.1:7860)")
    gradio_ui.launch(share=True)

    print("\n" + "=" * 50)
    print("INTERFACE LAUNCHED")
    print("=" * 50)

# --- STANDALONE WEB APP USING STREAMLIT ---
def run_streamlit_app():
    import streamlit as st

    st.set_page_config(page_title="Plant Disease Diagnosis", layout="wide", initial_sidebar_state="expanded")
    st.title("🌿 Plant Disease Diagnosis System")
    st.markdown("Upload an image of a plant leaf to diagnose potential diseases. "
                "The system uses a ResNet9 model to predict.")

    model_file = "final_plant_disease_model.pth"
    class_names_file = "class_names.txt"

    if not os.path.exists(class_names_file):
        default_classes = [
            "Apple___Apple_scab", "Apple___Black_rot", "Apple___Cedar_apple_rust", "Apple___healthy",
            "Background_without_leaves", "Blueberry___healthy", "Cherry___Powdery_mildew", "Cherry___healthy",
            "Corn___Cercospora_leaf_spot Gray_leaf_spot", "Corn___Common_rust", "Corn___Northern_Leaf_Blight", "Corn___healthy",
            "Grape___Black_rot", "Grape___Esca_(Black_Measles)", "Grape___Leaf_blight_(Isariopsis_Leaf_Spot)", "Grape___healthy",
            "Orange___Haunglongbing_(Citrus_greening)", "Peach___Bacterial_spot", "Peach___healthy",
            "Pepper,_bell___Bacterial_spot", "Pepper,_bell___healthy", "Potato___Early_blight", "Potato___Late_blight", "Potato___healthy",
            "Raspberry___healthy", "Soybean___healthy", "Squash___Powdery_mildew", "Strawberry___Leaf_scorch", "Strawberry___healthy",
            "Tomato___Bacterial_spot", "Tomato___Early_blight", "Tomato___Late_blight", "Tomato___Leaf_Mold",
            "Tomato___Septoria_leaf_spot", "Tomato___Spider_mites Two-spotted_spider_mite", "Tomato___Target_Spot",
            "Tomato___Tomato_Yellow_Leaf_Curl_Virus", "Tomato___healthy"
        ]
        try:
            with open(class_names_file, 'w') as f:
                for cls_name in default_classes:
                    f.write(cls_name + '\n')
        except Exception:
            pass

    @st.cache_resource
    def get_predictor_cached():
        print("Initializing predictor for Streamlit...")
        return PlantDiseasePredictor(model_path=model_file, class_names_path=class_names_file)

    predictor = get_predictor_cached()

    st.sidebar.header("ℹ️ About")
    st.sidebar.info(
        "This application uses a ResNet9 deep learning model to diagnose diseases in plant leaves. "
        "Upload a clear image of an individual leaf for best results."
    )

    # REMOVED EXAMPLE IMAGES PART FROM SIDEBAR
    # st.sidebar.header("💡 Try Example Images")
    # selected_example = st.sidebar.selectbox("Choose an example:", options=["None"] + list(example_images_streamlit.keys()))

    uploaded_file = st.file_uploader("📁 Upload Leaf Image", type=["jpg", "jpeg", "png"])

    image_to_process = None
    image_name = "Uploaded Image" # Default name

    if uploaded_file is not None:
        image_to_process = uploaded_file.getvalue()
        image_name = uploaded_file.name
    # REMOVED LOGIC FOR LOADING SELECTED EXAMPLE
    # elif selected_example != "None" and os.path.exists(example_images_streamlit[selected_example]):
    #     with open(example_images_streamlit[selected_example], "rb") as f:
    #         image_to_process = f.read()
    #     image_name = os.path.basename(example_images_streamlit[selected_example])

    if image_to_process:
        col1, col2 = st.columns([0.8, 1.2])
        pil_display_image = Image.open(io.BytesIO(image_to_process))

        with col1:
            st.subheader("🖼️ Uploaded Image")
            st.image(pil_display_image, caption=f"Input: {image_name}", use_column_width=True)

        with col2:
            st.subheader("🩺 Diagnosis Results")
            with st.spinner("🔍 Analyzing leaf image... This may take a moment."):
                vis_pil_image, result_text = generate_prediction_outputs(image_to_process, predictor.model, predictor.class_names, predictor.device)

            if vis_pil_image:
                st.image(vis_pil_image, use_column_width=True)
                lines = result_text.strip().split('\n')
                st.markdown(f"**{lines[0]}**")
                for line in lines[1:]:
                    parts = line.split(': ')
                    if len(parts) >= 2:
                        disease_info = parts[0]
                        probability_str = parts[1]
                        st.markdown(f"- {disease_info}: **{probability_str}**")
            else:
                st.error(result_text)

    with st.expander("📚 Supported Plants and Diseases", expanded=False):
        plants_diseases = {}
        for cls_name in predictor.class_names:
            plant, *disease_parts = cls_name.split('___')
            plant = plant.replace('_', ' ')
            disease = disease_parts[0].replace('_', ' ') if disease_parts else "Healthy"
            if plant not in plants_diseases: plants_diseases[plant] = []
            plants_diseases[plant].append(disease)

        for plant, diseases_list in sorted(plants_diseases.items()):
            st.markdown(f"**{plant}**: {', '.join(sorted(list(set(diseases_list))))}")


# Choose which implementation to run
if __name__ == '__main__':
    APP_CHOICE = "gradio"  # Change to "streamlit" to run the Streamlit app

    if APP_CHOICE.lower() == "gradio":
        run_gradio_app()
    elif APP_CHOICE.lower() == "streamlit":
        run_streamlit_app()
    else:
        print(f"Invalid APP_CHOICE: {APP_CHOICE}. Choose 'gradio' or 'streamlit'.")

PLANT DISEASE INTERACTIVE PREDICTOR (GRADIO)
Using device: cpu
Loaded 38 classes from class_names.txt
Model state loaded successfully from best_plant_disease_model.pth


C:\Users\pra72\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\gradio\interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated.Use `flagging_mode` instead.
  warnings.warn(



Launching Gradio interface... (Access it at the URL provided by Gradio, e.g., http://127.0.0.1:7860)
* Running on local URL:  http://127.0.0.1:7864

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.



INTERFACE LAUNCHED
